In [0]:
import re
import requests
import os
from pyspark.sql.functions import lit, current_timestamp

In [0]:
CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")
VOLUME = dbutils.widgets.get("VOLUME")
CITIES = dbutils.widgets.get("CITIES")
GET_DATA_URL = dbutils.widgets.get("GET_DATA_URL")
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
COUNTRY = dbutils.widgets.get("COUNTRY")

In [0]:
city_refined = re.sub(r"[^a-z0-9]+", "_", CITIES.lower()).strip("-")

In [0]:
city_volume_path = f"{VOLUME_PATH}/{city_refined}"

In [0]:
def get_page_html(url: str = GET_DATA_URL) -> str:
    """Fetch the Inside Airbnb data page HTML."""
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    return resp.text

In [0]:
def find_city_links(page_html: str, city: str) -> list:
    """Return the current data.insideairbnb.com download links from the city's section of the page.

    Each city has a section: <h3>City, State, Country</h3> followed by a table of
    file links. The section runs until the archived-data toggle / next city heading.
    """
    for m in re.finditer(r"<h3>([^<]+)</h3>", page_html, flags=re.IGNORECASE):
        if city.lower() not in m.group(1).lower():
            continue
        candidates = [
            x
            for x in (
                page_html.find("<h3", m.end()),
                page_html.find('class="showArchivedData"', m.end()),
            )
            if x != -1
        ]
        section_end = min(candidates) if candidates else len(page_html)
        return re.findall(r'href="(https://data\.insideairbnb\.com/[^"]+)"', page_html[m.end() : section_end])
    return []

In [0]:
def save_to_volume(url: str, dest: str) -> None:
    """Stream a remote file directly into the UC volume, overwriting any existing copy."""
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with requests.get(url, stream=True, timeout=600) as resp:
        resp.raise_for_status()
        with open(dest, "wb") as f:  # 'wb' overwrites any previous file
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)

In [0]:
def download_city_files(city: str, page_html: str) -> list:
    """Download every file currently listed for a city into the volume, overwriting existing copies.

    Returns a list of (city, file, source_url, bytes_written) tuples.
    """
    links = find_city_links(page_html, city)
    if not links:
        return [(city, "NOT FOUND ON PAGE", "", 0)]

    slug = re.sub(r"[^a-z0-9]+", "_", city.lower()).strip("-")
    results = []
    for url in links:
        name = url.rstrip("/").split("/")[-1]
        dest = f"{VOLUME_PATH}/{slug}/{name}"
        save_to_volume(url, dest)
        results.append((city, name, url, os.path.getsize(dest)))
    return results

In [0]:
def list_raw_files(volume_path: str = city_volume_path) -> list:
    #volume_path =f"{VOLUME_PATH}/{CITIES}"
    """Return absolute paths of all data files currently in the volume."""
    files = []
    for root, _, names in os.walk(volume_path):
        for name in names:
            files.append(os.path.join(root, name))
    return files

In [0]:
def load_file_to_table(src: str, catalog: str = CATALOG, schema: str = SCHEMA) -> str:
    """Load one raw file from the volume into a Delta table (overwrite mode).

    Table naming: <city-slug>_<file-stem>, e.g. new-york-city_listings -> tanmay_projects.project_1.new-york-city_listings
    """
    #volume_path =f"{VOLUME_PATH}/{CITIES}"
    rel = os.path.relpath(src, VOLUME_PATH)  # e.g. "new-york-city/listings.csv"
    city_slug, filename = rel.split(os.sep, 1)
    stem = os.path.splitext(filename)[0].replace(".", "_")
    if filename.endswith(".geojson"):  # avoid colliding with the neighbourhoods.csv table
        stem += "_geojson"
    table_name = f"{city_slug}_{stem}".replace("-", "_")
    full_table = f"{catalog}.{schema}.{table_name}"
    
    if filename.endswith(".csv") or filename.endswith(".csv.gz"):
        df = (
            spark.read.format("csv")
            .option("header", True)
            .option("inferSchema", True)
            .option("multiLine", True)
            .load(src)
        )
        df.select("*",lit(COUNTRY).alias("Country"),current_timestamp().alias("load_ts")).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(full_table)
    elif filename.endswith(".geojson"):
        df = spark.read.format("json").option("multiLine", True).load(src)
        df.select("*",lit(COUNTRY).alias("Country"),current_timestamp().alias("load_ts")).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(full_table)
    else:
        print(f"Skipping unsupported file type: {rel}")
        return None
    return full_table